## Missing Data Mechanisms

Missing data mechanisms describe **why values are missing** in a dataset. Understanding these mechanisms is essential because they determine **how missing values should be handled** and whether missingness may introduce **bias** into analysis or models.

---

### 1. Missing Completely at Random (MCAR)

**Definition:**  
Data are missing completely at random when the probability of a value being missing is **independent of both observed and unobserved data**. Indeed, the likelihood of missing value for all the variables is the same.

**What this means:**

- Missingness has no pattern.
- The missing values do not depend on any variable in the dataset.

**Implications:**

- Analysis remains unbiased.
- Simple approaches such as **dropping rows** or **basic imputation** are usually acceptable.

**Example:**

- A data collection system randomly crashes for a few minutes, causing random rows to be missing.

- Survey responses are lost due to a random network failure.

- Sensor readings are missing because of random hardware glitches unrelated to the measurements.

---

### 2. Missing at Random (MAR)

**Definition:**  
Data are missing at random when the probability of missingness depends on **observed variables**, but **not on the missing value itself**. In other words, probability of missing value depends on other variables (available information).

**What this means:**

- Missingness has a pattern.
- The pattern can be explained using available data.

**Implications:**

- Dropping rows may introduce bias.
- Imputation should use related observed variables.

**Example:**

- Income is missing more frequently for younger individuals, but age is observed.

- Medical test results are missing more often for patients from certain hospitals, and hospital ID is known.

- Credit score is missing more frequently for customers with low transaction history, which is recorded.

---

### 3. Missing Not at Random (MNAR)

**Definition:**  
Data are missing not at random when the probability of missingness depends on the **unobserved value itself**. Indeed, there is a reason for missing data.

**What this means:**

- Missingness is directly related to the missing data.
- The cause of missingness cannot be fully explained using observed variables.

**Implications:**

- Standard imputation methods may introduce strong bias.
- Requires domain knowledge or specialized modeling.

**Example:**

- Individuals with very high income choose not to report their income.

- Patients with severe symptoms skip follow-up medical tests.

- Users who are dissatisfied are less likely to leave feedback ratings.

| Mechanism | Depends on Observed Data | Depends on Missing Value | Risk of Bias |
| --------- | ------------------------ | ------------------------ | ------------ |
| MCAR      | No                       | No                       | Low          |
| MAR       | Yes                      | No                       | Medium       |
| MNAR      | Yes                      | Yes                      | High         |


---
## General Workflow for Missing values

1. Understand the source of missingness

- Data collection issue?

- User behavior?

- Sensor failure?

- Business rule?

2. Quantify missingness

- Percentage of missing values

- Patterns across time or groups

3. Choose a handling strategy

- Removal

- Imputation

- Flagging

- Modeling missingness explicitly
---


## Handling by Missing Data Mechanism

### 1. MCAR (Missing Completely at Random)

**Industry approach:**  
Often treated as noise.

**Common strategies:**

- Drop rows (if the missing rate is low)
- Mean / median / mode imputation

**Example (Industry):**

- Random packet loss in logs
- Occasional sensor glitches

**Risk:** Low  
**Complexity:** Low

---

### 2. MAR (Missing at Random)

**Industry approach:**  
Missingness is informative, but explainable.

**Common strategies:**

- Conditional imputation (e.g., regression, KNN)
- Group-based imputation
- Add a missing indicator feature

**Example (Industry):**

- Income missing for younger users
- Medical test missing for certain hospitals

**Best practice:**  
Impute the missing values **and** add a missing flag.

**Risk:** Medium  
**Complexity:** Medium

---

### 3. MNAR (Missing Not at Random)

**Industry approach:**  
Treated as high-risk and often escalated to domain experts.

**Common strategies:**

- Keep missing values as-is
- Add explicit missing indicators
- Use separate models for missing vs. non-missing cases
- Apply domain-informed imputation

**Example (Industry):**

- High-income users choosing not to report salary
- Fraud cases with missing documents

**Key rule:**  
Never blindly impute MNAR data.

**Risk:** High  
**Complexity:** High


---
### Missing Indicator (Very Important)
Best practice for MAR & MNAR as initial step.
```python
df['income_missing'] = df['income'].isna().astype(int)
```

### Constant / Sentinel Value Imputation

```python
df['age'] = df['age'].fillna(-1)
df['city'] = df['city'].fillna('Unknown')
```
---

### Group-Based Imputation (Conditional Mean)

```python
df['income'] = df.groupby('education')['income'] \
                  .transform(lambda x: x.fillna(x.median()))
```

---

### Forward / Backward Fill (Time Series)

```python
df['temperature'] = df['temperature'].ffill()
df['temperature'] = df['temperature'].bfill()
```

When to use:

- Time-series data
- Sensor readings
- Logs

---

### Median Imputation

from sklearn.impute import SimpleImputer

```python
imputer = SimpleImputer(strategy='median')
df['income'] = imputer.fit_transform(df[['income']])
```

---

### Regression Imputation

Predict missing values using other features.

```python
from sklearn.linear_model import LinearRegression

train = df[df['income'].notna()]
test  = df[df['income'].isna()]

model = LinearRegression()
model.fit(train[['age', 'education']], train['income'])

df.loc[df['income'].isna(), 'income'] = model.predict(
    test[['age', 'education']]
)
```

_Risk: Overconfident predictions_

---

### KNN Imputation

```python
from sklearn.impute import KNNImputer

imputer = KNNImputer(n_neighbors=5)
df[['age', 'income']] = imputer.fit_transform(df[['age', 'income']])

```

_Risk: Computationally expensive_

---

### Iterative / MICE Imputation (Industry Standard)

Models each variable conditionally.

```python
from sklearn.impute import IterativeImputer

imputer = IterativeImputer(random_state=0)
df_imputed = imputer.fit_transform(df)

```

Best for:

- MAR
- Structured tabular data

---

### Time-Series Interpolation

```python
df['value'] = df['value'].interpolate(method='linear')
```

Other methods:

- time
- polynomial
- spline

---

### Domain-Informed Imputation (MNAR)

When to use:

- High-risk features
- Regulated domains (health, finance)

Example:

- Missing income → treat as high income
- Missing test → assume worst case

```python
df['income'] = df['income'].fillna(df['income'].quantile(0.9))
```

---

### Missing Data Imputation Methods

| Method        | Use Case          | Risk     | Industry Usage    |
| ------------- | ----------------- | -------- | ----------------- |
| Mean / Median | MCAR              | Low      | Baseline          |
| Mode          | Categorical       | Low      | Common            |
| Constant      | Tree-based models | Low      | Very common       |
| Group-based   | MAR               | Medium   | Very common       |
| Forward Fill  | Time series       | Low      | Logs, sensors     |
| Missing Flag  | MAR / MNAR        | Very Low | Best practice     |
| Regression    | MAR               | Medium   | Moderate          |
| KNN           | MAR               | Medium   | Limited           |
| MICE          | MAR               | Low      | Industry standard |
| Domain-based  | MNAR              | High     | Expert-driven     |
